# Tutorial 3: The Residual Stream

**Prerequisites:** T01 (Transformers from Scratch), T02 (Hooks and Caching)

**What you'll learn:**
- Why the residual stream is the central object in mechanistic interpretability
- How to decompose the residual stream into per-component contributions
- How information accumulates layer by layer
- The "linear representation hypothesis" and what it means in practice

---

## The Residual Stream as Communication Bus

The key insight of the "mathematical framework for transformer circuits" (Elhage et al., 2021) is that the residual stream is a **shared communication channel**. Every component — attention heads and MLPs — reads from it and writes to it by **addition**:

```
residual = embed + pos_embed
residual = residual + attn_0(residual)   # Layer 0 attention adds
residual = residual + mlp_0(residual)    # Layer 0 MLP adds
residual = residual + attn_1(residual)   # Layer 1 attention adds
residual = residual + mlp_1(residual)    # Layer 1 MLP adds
logits = unembed(layernorm(residual))    # Final readout
```

Because everything is addition, the final residual stream is literally a **sum of all component outputs**:

```
residual_final = embed + pos_embed + attn_0_out + mlp_0_out + attn_1_out + mlp_1_out
```

This means we can ask: **how much does each component contribute to the final prediction?**

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig

cfg = HookedTransformerConfig(
    n_layers=3, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 42, 17, 88, 55])
print('Model ready: 3 layers, 4 heads, d_model=32')

## Decomposing the Residual Stream

Let's verify the additive structure by manually summing all contributions and checking they equal the final residual stream.

In [ ]:
logits, cache = model.run_with_cache(tokens)

# The individual contributions
embed = cache['blocks.0.hook_resid_pre']  # = embed + pos_embed
components = [embed]
component_names = ['embed+pos']

for l in range(cfg.n_layers):
    attn_out = cache[f'blocks.{l}.hook_attn_out']
    mlp_out = cache[f'blocks.{l}.hook_mlp_out']
    components.append(attn_out)
    components.append(mlp_out)
    component_names.append(f'L{l} Attn')
    component_names.append(f'L{l} MLP')

# Sum them all up
reconstructed = sum(components)

# Compare to actual final residual
final_resid = cache[f'blocks.{cfg.n_layers - 1}.hook_resid_post']
error = float(jnp.max(jnp.abs(reconstructed - final_resid)))
print(f'Max reconstruction error: {error:.2e}')
print(f'This confirms: residual_final = sum of all component outputs')
print()
print(f'Components ({len(components)} total):')
for name, comp in zip(component_names, components):
    norm = float(jnp.linalg.norm(comp[-1]))
    print(f'  {name:12s}: norm = {norm:.4f}')

## The Accumulated Residual Stream

Rather than looking at individual contributions, we often want to see the **running total** — the residual stream as it exists after each component has added its output. IRTK's `ActivationCache` has a method for this.

In [ ]:
# accumulated_resid gives the residual after each component
resid_stack = cache.accumulated_resid()  # [n_components, seq, d_model]
print(f'Accumulated residual shape: {resid_stack.shape}')
print(f'  = [{resid_stack.shape[0]} stages, {resid_stack.shape[1]} positions, {resid_stack.shape[2]} dims]')
print()

# Track the norm at the last position through the model
stage_names = ['embed']
for l in range(cfg.n_layers):
    stage_names.append(f'L{l} attn')
    stage_names.append(f'L{l} mlp')

print('Residual stream norm at last position, through the model:')
for i, name in enumerate(stage_names):
    if i < resid_stack.shape[0]:
        norm = float(jnp.linalg.norm(resid_stack[i, -1]))
        bar = '#' * int(norm * 10)
        print(f'  After {name:10s}: {norm:.4f} {bar}')

## Projecting the Residual Stream onto Token Directions

The unembedding matrix `W_U` maps the residual stream to logits. Column `t` of `W_U` is the **direction in residual-stream space that promotes token `t`**.

We can project the residual stream onto any token's direction to see how strongly it promotes that token at each stage. This is the intuition behind the **logit lens** (Tutorial T07).

In [ ]:
# Pick the model's top prediction at the last position
predicted_token = int(jnp.argmax(logits[-1]))
print(f'Model predicts token {predicted_token} at the last position')

# Get that token's direction in residual stream space
W_U = model.unembed.W_U  # [d_model, d_vocab]
token_dir = W_U[:, predicted_token]
token_dir_unit = token_dir / jnp.linalg.norm(token_dir)

# Project the accumulated residual stream onto this direction
projections = jnp.einsum('cd,d->c', resid_stack[:, -1, :], token_dir_unit)

print(f'\nProjection of residual onto token {predicted_token} direction:')
for i, name in enumerate(stage_names):
    if i < len(projections):
        proj = float(projections[i])
        direction = '+' if proj > 0 else '-'
        bar = direction * int(abs(proj) * 5)
        print(f'  After {name:10s}: {proj:+.4f} {bar}')

print(f'\nWatch how the projection grows — each component either pushes')
print(f'toward this token (positive) or away from it (negative).')

## Per-Component Contributions to the Logit

Since logits are linear in the residual stream (ignoring LayerNorm for now), and the residual stream is a sum of component outputs, we can **decompose the logit into per-component contributions**:

```
logit(token) = (embed · W_U[:, token]) + (attn_0_out · W_U[:, token]) + ... + bias
```

This is **direct logit attribution** — one of the most fundamental tools in mechinterp.

In [ ]:
# Direct logit attribution for the predicted token
pos = -1  # Last position

print(f'Logit attribution for token {predicted_token}:')
print(f'  (How much does each component contribute to this prediction?)\n')

total = 0.0
for name, comp in zip(component_names, components):
    # Project this component's output onto the token direction
    attribution = float(jnp.dot(comp[pos], W_U[:, predicted_token]))
    total += attribution
    sign = '+' if attribution > 0 else '-'
    bar = sign * int(abs(attribution) * 3)
    print(f'  {name:12s}: {attribution:+.4f} {bar}')

# Add bias
bias_contribution = float(model.unembed.b_U[predicted_token])
total += bias_contribution
print(f'  {"bias":12s}: {bias_contribution:+.4f}')

actual_logit = float(logits[pos, predicted_token])
print(f'\n  Sum:          {total:+.4f}')
print(f'  Actual logit: {actual_logit:+.4f}')
print(f'  (Difference due to LayerNorm, which we ignored)')

## Using IRTK's Residual Stream Tools

IRTK provides dedicated functions for residual stream analysis.

In [ ]:
from irtk.residual_stream import (
    cosine_similarity_to_unembed,
    residual_norm_by_layer,
    token_prediction_trajectory,
)

# How aligned is each layer's residual with the predicted token?
cos_sims = cosine_similarity_to_unembed(model, cache, predicted_token, pos=-1)
print(f'Cosine similarity to token {predicted_token} direction:')
for i, sim in enumerate(cos_sims):
    bar = '#' * int(abs(sim) * 20)
    label = 'embed' if i == 0 else f'layer {i-1}'
    print(f'  After {label:8s}: {sim:+.4f} {bar}')

print()

# Track the residual stream norm
norms = residual_norm_by_layer(cache, pos=-1)
print(f'Residual stream norms: {np.round(norms, 3)}')

In [ ]:
# Track how the top prediction changes across layers
trajectory = token_prediction_trajectory(model, tokens, pos=-1)
print(f'Prediction trajectory at position {len(tokens)-1}:')
for entry in trajectory:
    print(f'  After layer {entry["layer"]:2d}: '
          f'top token = {entry["top_token"]:3d}, '
          f'logit = {entry["top_logit"]:+.3f}, '
          f'prob = {entry["top_prob"]:.4f}')

print(f'\nNote how the model\'s prediction can change dramatically between layers.')
print(f'Understanding WHY it changes is the goal of mechanistic interpretability.')

## The Linear Representation Hypothesis

A key finding in mechinterp: **concepts are often represented as directions** in the residual stream. For example:

- There might be a "this token is a number" direction
- A "the next word should be plural" direction
- A "this is the subject of the sentence" direction

Components write along these directions to build up the model's understanding. The unembedding matrix reads these directions to make predictions.

Let's explore this by looking at how different positions have different representations.

In [ ]:
# Compare residual streams at different positions
final_resid = cache[f'blocks.{cfg.n_layers - 1}.hook_resid_post']

print('Pairwise cosine similarity between position residuals:')
for i in range(len(tokens)):
    row = []
    for j in range(len(tokens)):
        ri = final_resid[i]
        rj = final_resid[j]
        cos = float(jnp.dot(ri, rj) / (jnp.linalg.norm(ri) * jnp.linalg.norm(rj)))
        row.append(f'{cos:+.2f}')
    print(f'  pos {i}: [{"  ".join(row)}]')

print(f'\nEach position develops its own representation based on the tokens it')
print(f'has seen. Attention moves information between positions, and MLP layers')
print(f'transform the representation at each position independently.')

## Interference Between Components

A subtlety: because all components write to the same residual stream, their outputs can **interfere** with each other. Two components might push in opposite directions, partially cancelling each other out.

Understanding this interference is crucial for understanding the model's behavior.

In [ ]:
# Compute pairwise cosine similarity between component outputs
pos = -1
outputs = [comp[pos] for comp in components]

print(f'Pairwise cosine similarity between component outputs (pos {len(tokens)-1}):')
print(f'{"":14s}', end='')
for name in component_names:
    print(f'{name:>9s}', end='')
print()

for i, name_i in enumerate(component_names):
    print(f'{name_i:12s}  ', end='')
    for j in range(len(component_names)):
        oi = outputs[i]
        oj = outputs[j]
        ni = float(jnp.linalg.norm(oi))
        nj = float(jnp.linalg.norm(oj))
        if ni > 1e-10 and nj > 1e-10:
            cos = float(jnp.dot(oi, oj) / (ni * nj))
        else:
            cos = 0.0
        print(f'  {cos:+.2f}  ', end='')
    print()

print(f'\nComponents with negative cosine similarity are pushing in opposite')
print(f'directions — they may be "competing" to control the prediction.')

## Key Takeaways

1. **The residual stream is a sum** of all component outputs — this makes it decomposable
2. **Direct logit attribution** tells you how much each component contributes to a prediction
3. **Cosine similarity to unembed directions** tracks how the prediction forms across layers
4. **The linear representation hypothesis** suggests concepts are directions in residual stream space
5. **Component interference** means components can cancel or reinforce each other

The residual stream is the most important object in mechanistic interpretability because:
- It's where **all the information lives**
- It's **linearly decomposable** into component contributions
- The **unembedding matrix** provides a natural basis for interpreting directions

**Next: [T04 — Attention Heads Decoded](T04_attention_heads_decoded.ipynb)** — Deep dive into what attention heads do and how to analyze them.